# fastai Tabular: Airbnb NYC 2019, above or below average price

Binary classification: **1** if a listing is priced above the dataset average, otherwise **0**.

**Before running:** a GPU is optional, since tabular models train fine on CPU. You can enable one under *Runtime → Change runtime type → T4 GPU*. Then choose *Runtime → Run all*.

## 1. Dependencies
Colab already ships with `fastai`, `torch`, `pandas` and `kagglehub`, so nothing needs installing in the normal case.

Avoid running `pip install -U fastai` unless it is necessary. Upgrading can pull in a different `torch` build than Colab's CUDA setup expects, and that breaks the GPU. The cell below installs packages only if they are missing.

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure('fastai')
ensure('kagglehub')

import torch, fastai, pandas as pd
print('torch  :', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('fastai :', fastai.__version__)
print('pandas :', pd.__version__)

## 2. Get the dataset
Set `SOURCE` to one of these options:
- `'kagglehub'` (default): downloads the public Kaggle dataset directly, with no API key needed.
- `'upload'`: opens a file picker so you can upload `AB_NYC_2019.csv` from your computer.
- `'drive'`: reads the file from your Google Drive, at the path set in `DRIVE_PATH`.

In [ ]:
import os, glob

SOURCE = 'kagglehub'   # 'kagglehub' | 'upload' | 'drive'
DRIVE_PATH = '/content/drive/MyDrive/AB_NYC_2019.csv'

if SOURCE == 'kagglehub':
    import kagglehub
    folder = kagglehub.dataset_download('dgomonov/new-york-city-airbnb-open-data')
    CSV_PATH = glob.glob(os.path.join(folder, '**', 'AB_NYC_2019.csv'), recursive=True)[0]
elif SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))
elif SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    CSV_PATH = DRIVE_PATH
else:
    raise ValueError("SOURCE must be 'kagglehub', 'upload' or 'drive'")

print('Using:', CSV_PATH)

## 3. Load and clean

In [ ]:
import numpy as np
import pandas as pd
from fastai.tabular.all import *

set_seed(42, reproducible=True)

df = pd.read_csv(CSV_PATH)
print('Raw shape:', df.shape)

# $0 listings are data errors; the top 1% of prices would distort the mean
df = df[df['price'] > 0]
df = df[df['price'] <= df['price'].quantile(0.99)].copy()

# ID and free-text columns don't generalise
df = df.drop(columns=['id', 'name', 'host_id', 'host_name'])

# NaN here means the listing has no reviews
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# Expand last_review into date features (year, month, elapsed, ...)
df['last_review'] = pd.to_datetime(df['last_review'])
df = add_datepart(df, 'last_review', drop=True)

print('Clean shape:', df.shape)
df.head()

## 4. Binary target

In [ ]:
avg_price = df['price'].mean()
print(f'Average price: ${avg_price:.2f}')

df['above_avg'] = (df['price'] > avg_price).astype(int)
df = df.drop(columns=['price'])   # drop price so the model can't read the answer

df['above_avg'].value_counts(normalize=True).rename('class balance')

## 5. Columns, TabularPandas and DataLoaders

In [ ]:
dep_var = 'above_avg'
cont_names, cat_names = cont_cat_split(df, max_card=20, dep_var=dep_var)

for c in ['neighbourhood_group', 'neighbourhood', 'room_type']:
    if c in cont_names: cont_names.remove(c)
    if c not in cat_names: cat_names.append(c)

print('Categorical:', cat_names)
print('Continuous :', cont_names)

splits = RandomSplitter(valid_pct=0.2, seed=42)(range_of(df))

to = TabularPandas(
    df,
    procs=[Categorify, FillMissing, Normalize],
    cat_names=cat_names,
    cont_names=cont_names,
    y_names=dep_var,
    y_block=CategoryBlock(),
    splits=splits,
)

dls = to.dataloaders(bs=512)   # uses the GPU automatically if one is present
print('Device:', dls.device)
dls.show_batch(max_n=5)

## 6. Train

In [ ]:
learn = tabular_learner(
    dls,
    layers=[200, 100],
    metrics=[accuracy, F1Score(), RocAucBinary()],
)

lr = learn.lr_find().valley
print(f'Suggested learning rate: {lr:.2e}')

In [ ]:
learn.fit_one_cycle(8, lr)

## 7. Evaluate

In [ ]:
learn.show_results(max_n=10)

interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(5, 5))
interp.print_classification_report()

## 8. Predict

In [ ]:
# Predict one listing
row = df.iloc[0].drop(dep_var)
_, pred_class, probs = learn.predict(row)
label = 'ABOVE' if int(pred_class) == 1 else 'BELOW'
print(f'Sample listing predicted {label} average (p_above = {probs[1]:.3f})')

# Predict a batch of rows with test_dl
test_df = df.drop(columns=[dep_var]).sample(10, random_state=1)
test_dl = learn.dls.test_dl(test_df)
preds, _ = learn.get_preds(dl=test_dl)
test_df.assign(p_above=preds[:, 1].numpy().round(3))[['neighbourhood_group', 'room_type', 'p_above']]

## 9. Save and download the model

In [ ]:
learn.export('/content/airbnb_above_avg_price.pkl')
print('Saved to /content/airbnb_above_avg_price.pkl')

# Uncomment to download the model to your computer:
# from google.colab import files
# files.download('/content/airbnb_above_avg_price.pkl')

# To reload it later:
# learn2 = load_learner('/content/airbnb_above_avg_price.pkl')